# Оценка фиксированных состояний

Только вычисление метрик, без обучения и подбора по тесту.
По умолчанию повторяется опубликованная оценка 18 вариантов.
Нужны исходные веса и подготовленные данные в Drive.


In [ ]:
from pathlib import Path
import io, json, os, shutil, signal, subprocess, sys, zipfile
from google.colab import files, drive

PROJECT = Path('/content/rl_recsys')
PROJECT.mkdir(exist_ok=True)
uploaded = files.upload()  # dist/colab_bundle.zip
assert len(uploaded) == 1
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as z:
    for name in z.namelist():
        allowed = name in {'pyproject.toml', 'constraints_colab.txt'} or name.startswith(('src/', 'tests/', 'configs/', 'scripts/', 'notebooks/'))
        if not allowed or not (PROJECT / name).resolve().is_relative_to(PROJECT.resolve()):
            raise ValueError(name)
    z.extractall(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(PROJECT / 'constraints_colab.txt'), '-e', str(PROJECT) + '[train,test]'], check=True)
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
ENV = {**os.environ, 'PYTHONPATH': str(PROJECT / 'src'), 'PYTHONUNBUFFERED': '1'}

def run(command):
    with subprocess.Popen(list(map(str, command)), cwd=PROJECT, env=ENV,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            code = process.wait()
        except KeyboardInterrupt:
            if process.poll() is None:
                process.send_signal(signal.SIGINT)
            output, _ = process.communicate(timeout=60)
            print(output or '', end='')
            raise
        if code:
            raise subprocess.CalledProcessError(code, command)

def module(name, *args):
    run([sys.executable, '-m', name, *args])

run([sys.executable, '-c', "import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name())"])
run([sys.executable, '-m', 'pytest', '-q', '-ra'])


In [ ]:
BASE = Path('/content/drive/MyDrive/tbank_rl_recsys')
SELECTION = PROJECT / 'configs/final_selection.json'
OUTPUT = PROJECT / 'runs/final_evaluation'
BACKUP = BASE / 'evaluation_reproduction'
# Для нового обучения: BASE = .../rl_recsys_reproduction,
# SELECTION = BASE / 'fresh_selection.json', BACKUP = BASE / 'final_evaluation'.

def stage(name):
    module('recsys.final_evaluation', '--stage', name, '--base', BASE, '--runs', OUTPUT,
           '--backup', BACKUP, '--selection', SELECTION, '--max-minutes', '60', '--model-minutes', '5')
stage('inventory')
stage('preflight')


In [ ]:
try:
    stage('run')
finally:
    stage('export')


In [ ]:
progress = json.loads((BACKUP / 'progress.json').read_text())
print('Основное сравнение завершено:', progress['primary_complete'])
print('Все состояния оценены:', progress['available_models_complete'])
print('Количество оценок:', len(progress['completed']))
files.download(str(OUTPUT.parent / 'results_final_test_v1.zip'))
